# Model Merging & Storage Efficiency Analysis (LoRA vs. FFT)

## Overview
This notebook finalizes the **Adapter-Based Fine-Tuning** pipeline by consolidating the training artifacts into a deployment-ready state. The primary objective is to merge the lightweight **LoRA adapters** back into the base **NLLB-200** model to create a standalone translator that requires no external adapter dependencies during inference. Additionally, we conduct a quantitative analysis of storage efficiency, comparing the footprint of the LoRA-based approach against the traditional Full Fine-Tuning (FFT) strategy.

## Key Operations
1. **Model Loading & Precision Management:** Loaded the base NLLB model in `float16` on the CPU to ensure stable merging without memory overflows.
2. **Adapter Merging (merge_and_unload):** Mathematically folded the trained Low-Rank Adaptation matrices ($W_{new} = W_{base} + A \times B$) into the base model weights, eliminating inference latency overhead.
3. **Artifact Persistence:** Saved the final, merged model and tokenizer to a dedicated directory for future deployment.
4. **Footprint Analysis:** Calculated and compared the storage requirements of the two fine-tuning paradigms.

## Key Findings
* **LoRA Efficiency:** The LoRA adapter artifacts occupied only **299.48 MB**.
* **FFT Baseline:** The Full Fine-Tuned model required **2.33 GB**.
* **Conclusion:** The LoRA approach achieved a **87.43% reduction** in storage requirements while maintaining the translation quality established in previous evaluation steps.

## Requirements
* **Libraries:** `transformers`, `peft`, `torch`.
* **Paths:** Requires valid paths to the previously trained FFT model and LoRA adapter checkpoints.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import torch
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [ ]:
# --- 1. DEFINE PATHS ---
# Constants pointing to the various model artifacts:
# - Base Model: The original pre-trained NLLB model from Hugging Face.
# - FFT Model: Path where the Full Fine-Tuned model was saved (for size comparison).
# - LoRA Adapter: Path where the best Adapter checkpoints are stored.
# - Save Directory: Destination for the final merged standalone model.
BASE_MODEL_PATH = "facebook/nllb-200-distilled-600M"
FFT_MODEL_PATH = "/content/drive/MyDrive/Research_Paper_Publication/model/nllb-odia-german-translator_model_final_new_v1"
LORA_ADAPTER_PATH = "/content/drive/MyDrive/Research_Paper_Publication/model/lora-odia-german-translator-model-final-new-v1"
SAVE_DIRECTORY = "/content/drive/MyDrive/Research_Paper_Publication/model/NLLB_Odia_German_Best_Merged"

In [ ]:
print("🚀 Loading base model for merging (using float16)...")

# --- 2. LOAD BASE MODEL ---
# We load the base model specifically for the merging operation.
# CRITICAL NOTE: Merging typically works best when the base model is loaded in
# float16 or bfloat16 precision, rather than 4-bit/8-bit quantization.
# We map to "cpu" to prevent GPU Out-Of-Memory (OOM) errors, as merging duplicates
# weights temporarily in RAM.
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_PATH,
    dtype=torch.float16,
    device_map="cpu" # CPU is safer for merging to avoid OOM
)

🚀 Loading base model for merging (using float16)...


In [ ]:
# --- 3. LOAD ADAPTERS ---
# Attach the trained LoRA adapters to the base model structure.
# PeftModel handles the mapping of adapter weights to the correct base layers.
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_PATH)

In [ ]:
# --- 4. MERGE AND UNLOAD ---
print("🔄 Merging weights...")
# This is the core operation: It mathematically folds the LoRA matrices (A * B)
# into the original base model weights (W_new = W_old + A*B).
# The result is a standard architecture model with no separate adapter modules,
# offering the inference speed of the base model with the accuracy of the fine-tuned one.
merged_model = model.merge_and_unload()

🔄 Merging weights...


In [ ]:
# --- 5. SAVE FINAL MODEL ---
print(f"💾 Saving best model to {SAVE_DIRECTORY}...")
# Save the merged weights and config.
merged_model.save_pretrained(SAVE_DIRECTORY)

# Always save the tokenizer alongside the model to ensure the correct vocabulary
# and special tokens are used during deployment.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.save_pretrained(SAVE_DIRECTORY)

print("✅ Success! You now have a standalone best model.")

💾 Saving best model to /content/drive/MyDrive/Research_Paper_Publication/model/NLLB_Odia_German_Best_Merged...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✅ Success! You now have a standalone best model.


## Model Size Comparison

In [ ]:
# =========================================================
# ========= STORAGE FOOTPRINT ANALYSIS ====================
# =========================================================

def get_directory_size(directory):
    """
    Recursively calculates the total size of a directory in bytes.

    Args:
        directory (str): Path to the directory.

    Returns:
        int: Total size in bytes.
    """
    total_size = 0
    # os.walk yields a 3-tuple (dirpath, dirnames, filenames)
    for dirpath, dirnames, filenames in os.walk(directory):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            # Skip symbolic links to avoid double counting or infinite loops
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

def format_size(size_bytes):
    """
    Converts a file size in bytes into a human-readable string (e.g., "1.2 GB").

    Args:
        size_bytes (int): Size in bytes.

    Returns:
        str: Formatted string with appropriate unit.
    """
    if size_bytes == 0: return "0B"
    units = ("B", "KB", "MB", "GB", "TB")

    import math

    # Calculate the appropriate unit index (log base 1024)
    i = int(math.floor(math.log(size_bytes, 1024)))

    # Calculate the value in that unit
    p = math.pow(1024, i)
    s = round(size_bytes / p, 2)

    return f"{s} {units[i]}"

In [ ]:
# --- Calculate and Print Statistics ---
print("📊 Storage Footprint Comparison:")
print("-" * 40)

# 1. Measure Full Fine-Tuned (FFT) Model
try:
    fft_size = get_directory_size(FFT_MODEL_PATH)
    print(f"Full Fine-Tuned Model Size:   {format_size(fft_size)}")
except Exception as e:
    print(f"Error reading FFT path: {e}")

# 2. Measure LoRA Adapter
try:
    lora_size = get_directory_size(LORA_ADAPTER_PATH)
    print(f"LoRA Adapter Size:       {format_size(lora_size)}")
except Exception as e:
    print(f"Error reading LoRA path: {e}")

# 3. Calculate Compression Efficiency
# This demonstrates the storage advantage of LoRA
if 'lora_size' in locals() and 'fft_size' in locals():
    reduction = (1 - (lora_size / fft_size)) * 100
    print("-" * 40)
    print(f"📉 Storage Reduction:   {reduction:.2f}% smaller than Full Fine-Tuning")

📊 Storage Footprint Comparison:
----------------------------------------
Full Fine-Tuned Model Size:   2.33 GB
LoRA Adapter Size:       299.48 MB
----------------------------------------
📉 Storage Reduction:   87.43% smaller than Full Fine-Tuning
